In [1]:
pip install langchain groq tiktoken rapidocr-onnxruntime python-dotenv langchain-community

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from dotenv import load_dotenv
load_dotenv()



True

In [4]:
from langchain_community.document_loaders import TextLoader

C:\Users\Gaurav Sehgal\AppData\Local\Temp\ipykernel_19404\2929458509.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
C:\Users\Gaurav Sehgal\AppData\Roaming\Python\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
loader=TextLoader("/Users/Gaurav Sehgal/OneDrive/Desktop/Agentic_Rag/data/Agentic_Ai.txt",encoding="utf-8")
documents=loader.load()

In [6]:
documents

[Document(metadata={'source': '/Users/Gaurav Sehgal/OneDrive/Desktop/Agentic_Rag/data/Agentic_Ai.txt'}, page_content='AGENTIC AI - STUDY NOTES\n\n1.  What is Agentic AI? Agentic AI refers to AI systems that can\n    understand a goal, reason about the steps needed to achieve it, use\n    tools, observe results, and take further actions with limited human\n    intervention.\n\nA traditional LLM mainly generates a response to a prompt. An AI agent\ncan: - Understand a goal - Break the goal into tasks - Decide what\naction to take - Use external tools - Observe the result - Correct or\nchange its approach - Continue until the goal is completed\n\nBasic flow:\n\nUser Goal | v Reasoning / Planning | v Action / Tool Use | v Observation\n| +—-> Continue / Re-plan | v Final Result\n\n2.  LLM vs AI Agent\n\nLLM: - Primarily generates text. - Usually responds to a single\nprompt. - Does not inherently perform actions. - May have no persistent\nstate or tool access.\n\nAI Agent: - Uses an LLM as 

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [9]:
text_splitter=RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)

In [10]:
text_chunks=text_splitter.split_documents(documents)

In [11]:
!pip install faiss-cpu langchain-huggingface sentence-transformers
# pip install langchain_huggingface

Defaulting to user installation because normal site-packages is not writeable


In [13]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_groq import ChatGroq

print("Imports successful!")

Imports successful!


In [14]:
embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

In [15]:
vectorstore=FAISS.from_documents(text_chunks,embeddings)

In [16]:
query="What is Agentic AI?"

docs=vectorstore.similarity_search(query,k=4)

for doc in docs:
    print(doc.page_content)


AGENTIC AI - STUDY NOTES

1.  What is Agentic AI? Agentic AI refers to AI systems that can
    understand a goal, reason about the steps needed to achieve it, use
    tools, observe results, and take further actions with limited human
    intervention.
27. Simple Mental Model

Remember Agentic AI as:

GOAL | PLAN | ACT | OBSERVE | REFLECT / RE-PLAN | ACT AGAIN | SUCCESS

The key difference is that an agent does not only “answer”; it can
decide what to do next and interact with external systems to accomplish
a goal.
26. Interview Questions

Q1. What is Agentic AI? Answer: Agentic AI refers to AI systems capable
of pursuing goals by reasoning about tasks, selecting actions, using
tools, observing results, and adapting their approach.
AI Agent: - Uses an LLM as its reasoning component. - Can call tools
such as APIs, databases, search engines, calculators, or code
interpreters. - Maintains state or memory when required. - Can execute
multi-step workflows. - Can react to tool outputs and er

In [17]:
from langchain_core.prompts import ChatPromptTemplate

template="""You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use ten sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""

In [18]:
prompt=ChatPromptTemplate.from_template(template)

In [19]:
from langchain_core.output_parsers import StrOutputParser

In [20]:
output_parser=StrOutputParser()

In [ ]:
pip install --upgrade pip

Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/1.8 MB ? eta -:--:--
   ----------------------- ---------------- 1.0/1.8 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 1.8/1.8 MB 4.4 MB/s  0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 25.2
    Uninstalling pip-25.2:
      Successfully uninstalled pip-25.2
Note: you may need to restart the kernel to use updated packages.


In [21]:
from pydantic import SecretStr
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
GROQ_API_KEY_SECRET = SecretStr(GROQ_API_KEY) if GROQ_API_KEY is not None else None

In [22]:

from langchain_groq import ChatGroq


llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key= GROQ_API_KEY_SECRET
)


In [24]:
from langchain_core.runnables import RunnablePassthrough

In [25]:
retriever=vectorstore.as_retriever(search_kwargs={"k":4})

In [29]:
rag_chain=(
    {"context": retriever, "question": RunnablePassthrough()}| prompt | llm | output_parser
)

In [30]:
rag_chain.invoke("what is Agentic AI?")

'Agentic AI refers to AI systems that can understand a goal, reason about the steps needed to achieve it, use tools, observe results, and take further actions with limited human intervention. It involves a process of goal-setting, planning, acting, observing, reflecting, and re-planning to accomplish a goal. Agentic AI can decide what to do next and interact with external systems to achieve a goal. It can pursue goals by reasoning about tasks, selecting actions, using tools, observing results, and adapting its approach. An AI agent can use tools such as APIs, databases, and search engines, and maintain state or memory when required. It can execute multi-step workflows and react to tool outputs and errors. Agentic AI is capable of autonomous decision-making and action. It uses a reasoning component, such as a large language model (LLM), to make decisions. Overall, Agentic AI is a type of AI that can autonomously work towards achieving a goal.'